In [0]:
from pyspark.sql.functions import col, sum, avg, count, when, round, max, min, count_distinct
from pyspark.sql.window import Window

In [0]:
# BUSINESS KPIs FROM DATA CUBE

print("\n" + "="*60)
print("BUSINESS KPIs ANALYSIS (from Data Cube)")
print("="*60)

# KPI 1: Total Revenue (USD) 


total_revenue_usd = (
    spark.table("fact.sales_cube")
    .agg(
        sum("total_revenue_usd").alias("total_revenue_usd"),
        sum("total_revenue").alias("total_revenue_original_currency")
    )
)

print("\nKPI 1: Total Revenue (USD) ")
display(total_revenue_usd)

# KPI 2: Total Quantity Sold
total_quantity = (
    spark.table("fact.sales_cube")
    .agg(sum("total_quantity").alias("total_quantity_sold"))
)

print("\nKPI 2: Total Quantity Sold ")
display(total_quantity)

# KPI 3: Average Transaction Metrics 
avg_metrics = (
    spark.table("fact.sales_cube")
    .agg(
        round(avg("total_revenue_usd"), 2).alias("avg_revenue_usd_per_transaction"),
        round(avg("total_quantity"), 2).alias("avg_quantity_per_transaction"),
        sum("transaction_count").alias("total_transactions")
    )
)

print("\n KPI 3: Average Transaction Metrics ")
display(avg_metrics)

In [0]:
# KPI 4: Top 10 Customers by Revenue (USD)
top_customers = (
    spark.table("fact.sales_cube")
    .groupBy(
        col("customer_id"),
        col("customer_name"),
        col("customer_type"),
        col("customer_country")
    )
    .agg(
        sum("total_revenue_usd").alias("total_revenue_usd"),
        sum("total_quantity").alias("total_quantity"),
        sum("transaction_count").alias("transaction_count")
    )
    .orderBy(col("total_revenue_usd").desc())
    .limit(10)
)

print("\nKPI 4: Top 10 Customers by Revenue (USD)")
display(top_customers)

# KPI 5: Top 10 Products by Revenue (USD)
top_products = (
    spark.table("fact.sales_cube")
    .groupBy(
        col("product_id"),
        col("product_name"),
        col("category")
    )
    .agg(
        sum("total_revenue_usd").alias("total_revenue_usd"),
        sum("total_quantity").alias("total_quantity_sold"),
        sum("total_profit_usd").alias("total_profit_usd"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct")
    )
    .orderBy(col("total_revenue_usd").desc())
    .limit(10)
)

print("\n KPI 5: Top 10 Products by Revenue (USD) ")
display(top_products)

In [0]:
# KPI 6: Invoice Cancellation Rate 
invoice_cancellation = (
    spark.table("fact.fact_invoices")
    .groupBy()
    .agg(
        count("invoice_id").alias("total_invoices"),
        sum(when(col("invoice_status") == "cancelled", 1).otherwise(0)).alias("cancelled_invoices")
    )
    .withColumn(
        "cancellation_rate_pct",
        round((col("cancelled_invoices") / col("total_invoices")) * 100, 2)
    )
)

print("\n KPI 6: Invoice Cancellation Rate ")
display(invoice_cancellation)

# Breakdown by status

invoice_status_breakdown = (
    spark.table("fact.fact_invoices")
    .groupBy("invoice_status")
    .agg(
        count("invoice_id").alias("invoice_count"),
        sum("total_amount").alias("total_revenue")
    )
    .withColumn(
        "percentage",
        round((col("invoice_count") / sum("invoice_count").over(Window.partitionBy())) * 100, 2)
    )
    .orderBy(col("invoice_count").desc())
)

print("\nInvoice Status Breakdown:")
display(invoice_status_breakdown)

#  KPI 7: Discount Impact
discount_impact = (
    spark.table("fact.sales_cube")
    .agg(
        round(avg("avg_discount") * 100, 2).alias("overall_avg_discount_pct"),
        sum("total_revenue_usd").alias("total_revenue_usd_after_discount")
    )
)

print("\n KPI 7: Discount Impact (from Cube) ")
display(discount_impact)

# Discount by customer type (from cube)

discount_by_customer_type = (
    spark.table("fact.sales_cube")
    .groupBy(col("customer_type"))
    .agg(
        round(avg("avg_discount") * 100, 2).alias("avg_discount_pct"),
        sum("transaction_count").alias("transaction_count"),
        sum("total_revenue_usd").alias("total_revenue_usd")
    )
    .orderBy(col("avg_discount_pct").desc())
)

print("\nDiscount by Customer Type (from Cube):")
display(discount_by_customer_type)

In [0]:
#  KPI 8: Revenue by Region (USD)

revenue_by_region = (
    spark.table("fact.sales_cube")
    .groupBy(
        col("region_name"),
        col("region_code"),
        col("region_country")
    )
    .agg(
        sum("total_revenue_usd").alias("total_revenue_usd"),
        sum("total_quantity").alias("total_quantity"),
        sum("transaction_count").alias("transaction_count"),
        round(avg("total_revenue_usd"), 2).alias("avg_revenue_usd_per_transaction")
    )
    .orderBy(col("total_revenue_usd").desc())
)

print("\n KPI 8: Revenue by Region (USD) ")
display(revenue_by_region)

# Revenue by region and year
revenue_by_region_year = (
    spark.table("fact.sales_cube")
    .groupBy(
        col("region_name"),
        col("year")
    )
    .agg(
        sum("total_revenue_usd").alias("total_revenue_usd"),
        sum("transaction_count").alias("transaction_count")
    )
    .orderBy(col("year"), col("total_revenue_usd").desc())
)

print("\nRevenue by Region and Year (from Cube):")
display(revenue_by_region_year)

# Revenue by region and quarter
revenue_by_region_quarter = (
    spark.table("fact.sales_cube")
    .groupBy(
        col("region_name"),
        col("year"),
        col("quarter")
    )
    .agg(
        sum("total_revenue_usd").alias("total_revenue_usd"),
        sum("total_profit_usd").alias("total_profit_usd"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct")
    )
    .orderBy(col("year"), col("quarter"), col("total_revenue_usd").desc())
)

print("\nRevenue by Region and Quarter (from Cube):")
display(revenue_by_region_quarter.limit(20))

In [0]:
#KPI 9: Customer Purchase Frequency 

invoices_per_customer = (
    spark.table("fact.fact_invoices").alias("inv")
    .join(
        spark.table("dim.dim_customers").alias("cust"),
        col("inv.customer_key") == col("cust.customer_key"),
        "left"
    )
    .groupBy(
        col("cust.customer_id"),
        col("cust.customer_name"),
        col("cust.customer_type"),
        col("cust.country").alias("customer_country")
    )
    .agg(
        count("inv.invoice_id").alias("transaction_count"),
        sum("inv.total_amount").alias("total_revenue_usd"),
        round(avg("inv.total_amount"), 2).alias("avg_revenue_usd_per_transaction")
    )
    .orderBy(col("transaction_count").desc())
)

print("\nKPI 9: Top 20 Customers by Purchase Frequency ")
display(invoices_per_customer.limit(20))

# Customer segmentation by purchase frequency
customer_segments = (
    invoices_per_customer
    .withColumn(
        "customer_segment",
        when(col("transaction_count") >= 15, "High Frequency")
        .when(col("transaction_count") >= 8, "Medium Frequency")
        .otherwise("Low Frequency")
    )
    .groupBy("customer_segment")
    .agg(
        count("customer_id").alias("customer_count"),
        sum("total_revenue_usd").alias("total_revenue_usd"),
        round(avg("transaction_count"), 2).alias("avg_transactions_per_customer")
    )
    .orderBy(col("total_revenue_usd").desc())
)

print("\nCustomer Segmentation by Purchase Frequency (from Invoices):")
display(customer_segments)

# Average purchase frequency statistics
avg_purchase_frequency = (
    invoices_per_customer
    .agg(
        round(avg("transaction_count"), 2).alias("avg_transactions_per_customer"),
        max("transaction_count").alias("max_transactions_per_customer"),
        min("transaction_count").alias("min_transactions_per_customer"),
        count("customer_id").alias("total_customers")
    )
)

print("\nPurchase Frequency Statistics (from Invoices):")
display(avg_purchase_frequency)

In [0]:
print("\n" + "="*70)
print("           COMPREHENSIVE KPI SUMMARY DASHBOARD")
print("="*70)


#  REVENUE & PROFIT METRICS

financial_metrics = spark.table("fact.sales_cube").agg(
    sum("total_revenue_usd").alias("total_revenue_usd"),
    sum("total_profit_usd").alias("total_profit_usd"),
    sum("total_quantity").alias("total_quantity_sold"),
    sum("transaction_count").alias("total_transactions"),
    round(avg("profit_margin_pct"), 2).alias("avg_profit_margin_pct"),
    round(avg("avg_discount") * 100, 2).alias("avg_discount_pct")
).collect()[0]


#  INVOICE & CUSTOMER METRICS

invoice_customer_metrics = (
    spark.table("fact.fact_invoices").alias("inv")
    .join(spark.table("dim.dim_customers").alias("cust"),
          col("inv.customer_key") == col("cust.customer_key"), "left")
    .agg(
        # Invoice metrics
        count("inv.invoice_id").alias("total_invoices"),
        sum(when(col("inv.invoice_status") == "cancelled", 1).otherwise(0)).alias("cancelled_invoices"),
        
        # Customer metrics
        count_distinct("cust.customer_id").alias("total_customers"),
        round(count("inv.invoice_id") / count_distinct("cust.customer_id"), 2).alias("avg_invoices_per_customer")
    )
    .withColumn("cancellation_rate_pct", 
                round((col("cancelled_invoices") / col("total_invoices")) * 100, 2))
).collect()[0]


# SECTION 3: TOP PERFORMERS

top_customer = spark.table("fact.sales_cube").groupBy("customer_name").agg(
    sum("total_revenue_usd").alias("revenue")
).orderBy(col("revenue").desc()).first()

top_product = spark.table("fact.sales_cube").groupBy("product_name").agg(
    sum("total_revenue_usd").alias("revenue")
).orderBy(col("revenue").desc()).first()

top_region = spark.table("fact.sales_cube").groupBy("region_name").agg(
    sum("total_revenue_usd").alias("revenue")
).orderBy(col("revenue").desc()).first()


#  PRODUCT & REGION COUNTS

product_count = spark.table("dim.dim_products").count()
region_count = spark.table("dim.dim_regions").count()


kpi_data = [
    # Financial Performance

    ("Total Revenue (USD)", f"${financial_metrics['total_revenue_usd']:,.2f}", "Total sales revenue"),
    ("Total Profit (USD)", f"${financial_metrics['total_profit_usd']:,.2f}", "Revenue minus costs"),
    ("Average Profit Margin", f"{financial_metrics['avg_profit_margin_pct']:.2f}%", "Profitability percentage"),
    ("Average Discount", f"{financial_metrics['avg_discount_pct']:.2f}%", "Average discount given"),
    ("", "", ""),
    
    # Sales Volume
    ("SALES VOLUME", "", ""),
    ("Total Units Sold", f"{financial_metrics['total_quantity_sold']:,}", "Total quantity sold"),
    ("Total Transactions", f"{financial_metrics['total_transactions']:,}", "Total line items"),
    ("Total Invoices", f"{invoice_customer_metrics['total_invoices']:,}", "Total invoices issued"),
    ("", "", ""),
    
    # Customer Insights
    ("CUSTOMER INSIGHTS", "", ""),
    ("Total Customers", f"{invoice_customer_metrics['total_customers']:,}", "Unique customers"),
    ("Avg Invoices per Customer", f"{invoice_customer_metrics['avg_invoices_per_customer']:.2f}", "Purchase frequency"),
    ("Cancellation Rate", f"{invoice_customer_metrics['cancellation_rate_pct']:.2f}%", "% of cancelled invoices"),
    ("Cancelled Invoices", f"{invoice_customer_metrics['cancelled_invoices']:,}", "Total cancellations"),
    ("", "", ""),
    
    # Catalog
    ("PRODUCT CATALOG", "", ""),
    ("Total Products", f"{product_count:,}", "Unique products"),
    ("Total Regions", f"{region_count:,}", "Sales regions"),
    ("", "", ""),
    
    # Top Performers
    ("TOP PERFORMERS", "", ""),
    ("Best Customer", top_customer['customer_name'], f"${top_customer['revenue']:,.2f} revenue"),
    ("Best Product", top_product['product_name'], f"${top_product['revenue']:,.2f} revenue"),
    ("Best Region", top_region['region_name'], f"${top_region['revenue']:,.2f} revenue")
]

# Create and display summary table
kpi_summary = spark.createDataFrame(kpi_data, ["KPI", "Value", "Description"])

print("\n")
display(kpi_summary)

print("\n" + "="*70)

print("="*70)